In [1]:
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np

In [2]:
base_path = "../distractor_results"
score_path = "../distractor_results"
template = "Instruct-Query"
use_lang_specific_prompts=False
k = 2
models = [
          "Qwen__Qwen3-Embedding-0.6B",
          "__flash__project_462001491__models__v1-20260828-095152__checkpoint-18000",
          "microsoft__harrier-oss-v1-0.6b",
          "intfloat__multilingual-e5-large-instruct",
          "google__embeddinggemma-300m"
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000",
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-4000",
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-6000",
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-12000",
          ]
filter_prompts=False
dataset = "arcchallenge" #"tatoeba:fra-eng" #"mteb__tatoeba-bitext-mining:fin-eng" #"mteb__ARCChallenge"#"mteb__multi-hatecheck:eng" #"mteb__ARCChallenge" #"mteb__tatoeba-bitext-mining:ara-eng" #"mteb__multi-hatecheck:eng" #"mteb__reddit-clustering" #"mteb__stsbenchmark-sts" #"mteb__tatoeba-bitext-mining:fin-eng"
split = "test"
score= f"ndcg@{k}" # "ndcg@10"#"F1" #"Accuracy" #"V-score" # "average_precision" 
subsplit=""
path = lambda model: f"{base_path}/{model}/{dataset}/{split}/{template}_template/"
path_scores = lambda model: f"{score_path}/{model}/{dataset}/{split}/{template}_template/"

In [3]:
# Retrieval prompt
prompts_retrieval = ["Given a question, retrieve the passage that best answers it.",
            "Retrieve.",
            "Find the most relevant passage that directly answers the question.",
            "Given a question, find a related document.",
            "Retrieve the answer to the question.",
            "Retrieve text based on user query.",
            "Given a question, retrieve Wikipedia passages that answer the question.",
            ]
prompts_sts = ["Retrieve semantically similar text.",
            "Retrieve a similar passage.",
            "Represent this sentence for a natural language understanding task.",
            "Group passages based on semantic similarity.",
            ]
prompts_finetune = [ "Given a question, retrieve Wikipedia passages that answer the question.",
            "Given a question, retrieve questions that are semantically equivalent to the given question."]

In [4]:

def construct_df(model, show=False):
    scores_path= path_scores(model)+f"prompt_eval_k{k}.json"
    with open(scores_path) as f:
        scores = json.load(f)
    scores_path2= path_scores(model)+f"prompt_eval_with_paraphrase_distractors_k{k}.json"
    with open(scores_path2) as f:
        scores2 = json.load(f)
    with open(path(model)+f"prompt_geometry_{k}_false_positives.json") as f:
        data1 = json.load(f)
    df_scores = pd.DataFrame.from_dict(scores).T
    df_scores2 = pd.DataFrame.from_dict(scores2).T
    df_all_scores = df_scores.merge(df_scores2, suffixes=("", "_distracted"), on='prompt_text')
    df_angle = pd.DataFrame.from_dict(data1).T
    df = df_all_scores.merge(df_angle, on='prompt_text')
    if filter_prompts:
        prompts = prompts_retrieval+prompts_finetune+prompts_sts+["NO_PROMPT", "EMPTY"]
        df = df[df["prompt_text"].isin(prompts)]
    if show: display(df.head())
    return df

_ = construct_df(models[-1], show=True)

,prompt_text,recall@2,ndcg@2,recall@2_distracted,ndcg@2_distracted,example_text,sim_q2a,sim_q2pq,sim_pq2a,sim_q2a_euc,...,sim_pq2a_euc,chord_similarity,sim_improvement,knn_retention,hard_neg_sim_change_mean,hard_neg_sim_change_max,hard_neg_angulation_mean,hard_neg_angulation_max,paraphrase_neg_sim_change_max,paraphrase_neg_sim_change_mean
0,NO_PROMPT,"{'mean': 0.027303754266211604, 'std': 0.162967...","{'mean': 0.02352487802291595, 'std': 0.1434825...","{'mean': 0.005119453924914676, 'std': 0.071366...","{'mean': 0.0032300158032668473, 'std': 0.04502...",An astronomer observes that a planet rotates f...,"{'mean': 0.6124830139934406, 'std': 0.11401522...","{'mean': 1.0000000374309033, 'std': 6.59266330...","{'mean': 0.6124830139934406, 'std': 0.11401522...","{'mean': 0.8704780806046704, 'std': 0.13153699...",...,"{'mean': 0.8704780806046704, 'std': 0.13153699...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'...","{'mean': 0.019340159271899884, 'std': 0.077927...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'...","{'mean': 0.0, 'std': 0.0, 'median': 0.0, 'q25'..."
1,EMPTY,"{'mean': 0.026450511945392493, 'std': 0.160470...","{'mean': 0.022356729348488865, 'std': 0.138830...","{'mean': 0.00938566552901024, 'std': 0.0964239...","{'mean': 0.006866414700146468, 'std': 0.072317...",Instruct: \nQuery: An astronomer observes that...,"{'mean': 0.6124830139934406, 'std': 0.11401522...","{'mean': 0.8627401970759186, 'std': 0.02983556...","{'mean': 0.6391955433546886, 'std': 0.08351589...","{'mean': 0.8704780806046704, 'std': 0.13153699...",...,"{'mean': 0.8439473582005745, 'std': 0.09675678...","{'mean': 0.34963398158181547, 'std': 0.0994567...","{'mean': 0.026712529361248016, 'std': 0.053730...","{'mean': 0.018486916951080772, 'std': 0.083415...","{'mean': 0.005174534643265981, 'std': 0.045179...","{'mean': -0.004767795547893836, 'std': 0.04689...","{'mean': -0.3334929593091786, 'std': 0.0980469...","{'mean': -0.3592435613897575, 'std': 0.0995503...","{'mean': 0.07840821121536425, 'std': 0.0398420...","{'mean': 0.07840821121536425, 'std': 0.0398420..."
2,Identify categories in user passages.,"{'mean': 0.01877133105802048, 'std': 0.1357164...","{'mean': 0.01625208022915671, 'std': 0.1199936...","{'mean': 0.00938566552901024, 'std': 0.0964239...","{'mean': 0.006866414700146468, 'std': 0.072317...",Instruct: Identify categories in user passages...,"{'mean': 0.6124830139934406, 'std': 0.11401522...","{'mean': 0.7854416017951412, 'std': 0.03555878...","{'mean': 0.5997386715365352, 'std': 0.06172461...","{'mean': 0.8704780806046704, 'std': 0.13153699...",...,"{'mean': 0.8921246168047902, 'std': 0.06809133...","{'mean': 0.3431800742547819, 'std': 0.09716482...","{'mean': -0.012744342456905509, 'std': 0.07315...","{'mean': 0.022753128555176336, 'std': 0.096653...","{'mean': 0.049554608851264365, 'std': 0.066286...","{'mean': 0.03898809023786323, 'std': 0.0676350...","{'mean': -0.3307053521132484, 'std': 0.0960088...","{'mean': -0.35242215576724695, 'std': 0.096128...","{'mean': 0.14389096007208776, 'std': 0.0475745...","{'mean': 0.14389096007208776, 'std': 0.0475745..."
3,Classify user passages.,"{'mean': 0.012798634812286689, 'std': 0.112404...","{'mean': 0.010279383983422919, 'std': 0.092651...","{'mean': 0.004266211604095563, 'std': 0.065176...","{'mean': 0.0033214925432716487, 'std': 0.05209...",Instruct: Classify user passages.\nQuery: An a...,"{'mean': 0.6124830139934406, 'std': 0.11401522...","{'mean': 0.7889347038781683, 'std': 0.03676739...","{'mean': 0.6159061641009594, 'std': 0.06156328...","{'mean': 0.8704780806046704, 'std': 0.13153699...",...,"{'mean': 0.8737239455708872, 'std': 0.06924045...","{'mean': 0.3698942129891175, 'std': 0.08958263...","{'mean': 0.00342315010751880

In [5]:

def plot(df, x, y="score", colors=None, sizes=None, title="", legend_title=None, x_min=None, x_max=None, y_min=None, y_max=None, return_fig=False, add_line=False):
    if colors is None:
        colors = y
    if sizes is None:
        sizes = y
    
    # legend title that explains formatting
    if legend_title is None:
        legend_title = f"colors:{colors}, size:{sizes}"

    # Normalize scores for marker size
    min_size, max_size = 10, 30
    try:
        # parse the value from dictionary
        df["sizes"] = df[sizes].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
        ranks = df["sizes"].rank(method='average')
    except:   # for non-dict format: i.e. prompt_label or score
        ranks = df[sizes].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )
    #print(df["sizes"])

    #print(marker_sizes)
    y_vals = df[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    #y_err = df[y].apply(lambda d: float(d['std']) if isinstance(d, dict) else float(d[1]) if isinstance(d, list) else 0.0)
    
    y_err = []
    for line in df[y]:
        if isinstance(line, dict):
            if "std" in line.keys():
                y_err.append(float(line["std"]))
            elif "confidence_interval" in line.keys():
                y_err.append(float(max(line["confidence_interval"])))
            else:
                y_err.append(0.0)
        else:
            y_err.append(0.0)
        

    x_vals = df[x].apply(lambda d: float(d['mean']))
    x_err  = df[x].apply(lambda d: float(d['std']))
    
    symbols = ["circle" if p not in prompts_finetune else "x" for p in df["prompt_text"]]

    # Create figure
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            mode='markers',
            x=x_vals,
            y=y_vals,
            #error_x=dict(type='data', array=x_err, visible=True, color='lightgray'),  # optional std bars
            #error_y=dict(type='data', array=y_err, visible=True, color='lightgray'),
            marker=dict(
                size=marker_sizes,
                symbol=symbols,
            #    colorscale='Cividis',
                color=df[colors].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d)),
                colorbar=dict(title=f"C:{colors} S:{sizes}"),
                showscale=True,
            ),
            text=df['prompt_text'],
            hovertemplate=(
                '<b>Prompt:</b> %{text}<br>'
                '<b>X (distance):</b> %{x:.4f}<br>'
                '<b>Y (score):</b> %{y:.4f}<br>'
            ),
        ),
    )


    # Add one trace per alpha value

    fig.update_layout(
        title = title,
        xaxis_title=x,#'Cos-distance compared to Q-A line',
        yaxis_title=y,#'Prompt performance',
        height=600,
        width=1000,
        template="none",
    )
    fig.update_layout(legend_title_text=legend_title)
    fig.update_xaxes(range=[x_min, x_max])
    fig.update_yaxes(range=[y_min, y_max], autorange=False)

    if add_line:
        for (x0, y0), (x1, y1) in add_line:
            fig.add_shape(
                type="line",
                x0=x0, y0=y0,                  # Starting point (adjust to match your axis limits if needed)
                x1=x1, y1=y1,                  # Ending point
                xref="x", yref="y",          # Binds coordinates to the data scale
                xsizemode="scaled",          # Keeps the slope relative to the axes
                ysizemode="scaled",
                line=dict(color="Red", width=2, dash="dash") # Optional styling
            )


    if return_fig:
        return fig
    else:
        fig.show()

In [6]:
dfs = {}
for m in models:
    try:
        dfs[m] = construct_df(m)
    except Exception as e:
        print(f"Cannot construct results for {m}")
        print(e)

In [7]:

for m in dfs.keys():
    df = dfs[m]
    #print(df.columns)
    x = "paraphrase_neg_sim_change_mean"
    y = f"recall@{k}_distracted"
    x_min=-0.1
    x_max = 0.5
    plot(df, x=x, y=y, title=f"{dataset}: {m}", x_min=x_min, x_max = x_max)
    


In [8]:
for m in dfs.keys():
    df = dfs[m]
    #print(df.columns)
    sim_QA = df["sim_q2a_euc"][0]["mean"]  # this is the max value we can have
    x = "sim_q2pq_euc"#"hard_neg_sim_change_mean"
    y = "sim_pq2a_euc"#"paraphrase_neg_sim_change_mean" 
    sizes=f"recall@{k}_distracted"
    colors = f"recall@{k}_distracted"
    x_min=None
    x_max = None
    y_min=None
    y_max = None
    plot(df, x=x, y=y, sizes=sizes, colors=colors, title=f"{dataset}: {m}", x_min=x_min, x_max = x_max, y_min= y_min, y_max = y_max, add_line=[((0,0), (1,1)), ((sim_QA, 0), (0, sim_QA))])

In [9]:

def plot_paired_differences(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    hover_col: str = None,
    df1_name: str = "Group A",
    df2_name: str = "Group B",
    df1_color: str = "royalblue",
    df2_color: str = "tomato",
    line_color: str = "gray",
    x="displacement",
    y="ndcg@10"
    ):
    """
    Plot paired (x, y) points from two DataFrames, connected by dashed lines.

    Parameters
    ----------
    df1, df2    : DataFrames with columns 'x' and 'y' (same length, rows are paired)
    hover_col   : Optional column name in both DataFrames to show as hover text
    df1_name    : Legend label for df1 points
    df2_name    : Legend label for df2 points
    df1_color   : Marker color for df1
    df2_color   : Marker color for df2
    line_color  : Color of the dashed connector lines
    """
    assert len(df1) == len(df2), "DataFrames must have the same number of rows."

    fig = go.Figure()
    # for some data, we need to parse the column (from dict or tuple)
    y_vals1 = df1[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    y_vals2 = df2[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals1 = df1[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals2 = df2[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))

    # Connectroe lines
    # Interleave (x1, x2, None) for each pair so plotly draws separate segments
    line_x, line_y = [], []
    for x1, x2, y1, y2 in zip(x_vals1, x_vals2, y_vals1, y_vals2):
        line_x += [x1, x2, None]
        line_y += [y1, y2, None]

    fig.add_trace(go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        line=dict(dash="dash", color=line_color, width=1.5),
        hoverinfo="skip",
        showlegend=False,
    ))

    # Hover template
    def make_hover(df, group_name):
        if hover_col and hover_col in df.columns:
            return (
                df[hover_col].tolist(),
                f"<b>{group_name}</b><br>"
                f"x: %{{x}}<br>y: %{{y}}<br>"
                f"{hover_col}: %{{text}}<extra></extra>",
            )
        return (None, f"<b>{group_name}</b><br>x: %{{x}}<br>y: %{{y}}<extra></extra>")
    
    # First data
    colors1 = [df1_color if p in prompts_retrieval else "lightblue" for p in df1["prompt_text"]]
    symbols = ["circle" if p not in prompts_finetune else "x" for p in df1["prompt_text"]]
    text1, htemplate1 = make_hover(df1, df1_name)
    fig.add_trace(go.Scatter(
        x=x_vals1, y=y_vals1,
        mode="markers",
        name=df1_name,
        marker=dict(color=colors1, size=10, line=dict(width=1, color="white"), symbol=symbols),
        text=text1,
        hovertemplate=htemplate1,
    ))

    # Second data
    colors2 = [df2_color if p in prompts_retrieval else "lightpink" for p in df2["prompt_text"]]
    symbols = ["circle" if p not in prompts_finetune else "x" for p in df2["prompt_text"]]
    text2, htemplate2 = make_hover(df2, df2_name)
    fig.add_trace(go.Scatter(
        x=x_vals2, y=y_vals2,
        mode="markers",
        name=df2_name,
        marker=dict(color=colors2, size=10, line=dict(width=1, color="white"), symbol=symbols),
        text=text2,
        hovertemplate=htemplate2,
    ))

    fig.update_layout(
        xaxis_title=x,
        yaxis_title=y,
        height=600,
        width=1000,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white",
    )

    #fig.show()
    return fig


In [10]:
def plot_difference(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    hover_col: str = None,
    df1_name: str = "Group A",
    df2_name: str = "Group B",
    df1_color: str = "royalblue",
    df2_color: str = "tomato",
    line_color: str = "gray",
    x="displacement",
    y="ndcg@10",
    sizes=None,
    ):
    if sizes is None:
        sizes=y
    
    assert len(df1) == len(df2), "DataFrames must have the same number of rows."

    fig = go.Figure()
    # for some data, we need to parse the column (from dict or tuple)
    y_vals1 = df1[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    y_vals2 = df2[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals1 = df1[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals2 = df2[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals_diff = np.array(x_vals2)-np.array(x_vals1)
    y_vals_diff = np.array(y_vals2)-np.array(y_vals1)
    
    # Hover template
    def make_hover(df, group_name):
        if hover_col and hover_col in df.columns:
            return (
                df[hover_col].tolist(),
                f"<b>{group_name}</b><br>"
                f"x: %{{x}}<br>y: %{{y}}<br>"
                f"{hover_col}: %{{text}}<extra></extra>",
            )
        return (None, f"<b>{group_name}</b><br>x: %{{x}}<br>y: %{{y}}<extra></extra>")

    # Normalize scores for marker size
    min_size, max_size = 8, 32
    df1["sizes"] = df1[sizes].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    ranks = df1["sizes"].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )
    # First data
    colors = [df2_color if p in prompts_retrieval else "lightpink" for p in df2["prompt_text"]]
    symbols = ["circle" if p not in prompts_finetune else "x" for p in df2["prompt_text"]]
    text, htemplate = make_hover(df1, df1_name)
    fig.add_trace(go.Scatter(
        x=x_vals_diff, y=y_vals_diff,
        mode="markers",
        name="Difference",
        marker=dict(color=colors, size=marker_sizes, line=dict(width=1, color="white"), symbol=symbols),
        text=text,
        hovertemplate=htemplate,
    ))


    fig.update_layout(
        xaxis_title=f"Difference in {x}",
        yaxis_title=f"Difference in {y}",
        height=600,
        width=1000,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white",
    )

    #fig.show()
    return fig

In [11]:
x = "paraphrase_neg_sim_change_mean"
y = f"recall@{k}_distracted"

fig = plot_paired_differences(
                            dfs["Qwen__Qwen3-Embedding-0.6B"], 
                            #dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000"], 
                            dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-18000"], 
                            hover_col="prompt_text", 
                            x=x,
                            y=y,
                            df1_name="Qwen3-Embedding-0.6B", 
                            df2_name="Finetuning checkpoint 18k")
fig.show()

fig = plot_difference(
                    dfs["Qwen__Qwen3-Embedding-0.6B"], 
                    #dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000"], 
                    dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-18000"],
                    hover_col="prompt_text",
                    x=x,
                    y=y)
fig.show()